# Reporter mAP histogram

Per-reporter distinctiveness across genes: a boxplot of the raw per-gene distinctiveness values for each reporter, and a histogram of the per-reporter mean distinctiveness. Both reference the `all_combined` row from the same CSV (the median for the boxplot, the mean for the histogram).

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import pdist

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42

FIGURES_DIR = Path("../../output/figure_3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV path

Raw per-gene distinctiveness for each reporter, with one extra column `all_combined` representing the all-reporters-combined embedding. Curated into `../../data/figures/figure_3/` (see README).

In [ ]:
CSV_PATH = Path("../../data/figures/figure_3/gene_reporter_distinctiveness_all.csv")

## Load

`df` is genes × reporters; `all_combined` is the all-reporters-combined column popped off so it can be plotted as a separate reference line.

In [ ]:
df = pd.read_csv(CSV_PATH, index_col="gene")
all_combined = df.pop("all_combined")
print(f"reporters: {df.shape[1]}, genes: {df.shape[0]}")
df.head()

## Per-reporter summary statistics

Mean / median / std per reporter, sorted by median distinctiveness (descending). The boxplot below uses this ordering.

Also written to `reporter_distinctiveness_summary.csv` in the output dir (SI table), with a `rank` column (1 = highest median) and the all-reporters-combined embedding appended as a final `all_combined` reference row.

In [ ]:
stats = pd.DataFrame({
    "mean":   df.mean(),
    "median": df.median(),
    "std":    df.std(),
}).sort_values("median", ascending=False)

# SI table: the same summary with the all-reporters-combined embedding appended as a
# reference row, and a rank column (1 = highest median, matching the sort order above;
# blank for `all_combined`, which is a reference rather than a ranked reporter).
# `stats` itself stays reporter-only — the boxplot and histogram below index it against
# the columns of `df`.
combined_row = pd.DataFrame(
    {
        "mean":   [all_combined.mean()],
        "median": [all_combined.median()],
        "std":    [all_combined.std()],
    },
    index=["all_combined"],
)
stats_out = pd.concat([stats, combined_row])
stats_out.insert(0, "rank", list(range(1, len(stats) + 1)) + [pd.NA] * len(combined_row))
stats_out.index.name = "reporter"

STATS_CSV = FIGURES_DIR / "reporter_distinctiveness_summary.csv"
stats_out.to_csv(STATS_CSV)
print(f"wrote {STATS_CSV}  ({len(stats_out)} rows)")

stats_out

## Boxplot — per-reporter distinctiveness across genes

One box per reporter, ordered by median distinctiveness. Dashed red line is the median distinctiveness of the all-reporters-combined embedding.

In [ ]:
order = stats.index.tolist()
all_combined_median = all_combined.median()

fig, ax = plt.subplots(figsize=(18, 6))
ax.boxplot(
    [df[col].values for col in order],
    tick_labels=order,
    showfliers=True,
    flierprops=dict(marker=".", markersize=2, alpha=0.3),
)
ax.axhline(
    all_combined_median,
    linestyle="--",
    color="red",
    label=f"all_combined median = {all_combined_median:.3f}",
)
ax.set_ylabel("distinctiveness")
ax.set_xlabel("reporter")
ax.set_title("Per-reporter distinctiveness across genes")
ax.tick_params(axis="x", rotation=90)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "reporter_distinctiveness_boxplot.svg", bbox_inches="tight")
plt.show()

## Histogram — distribution of per-reporter mean distinctiveness

Steelblue bars are individual reporters (height = count); the crimson bar marks the all-reporters-combined embedding (height = 1, width = histogram bin width) at its mean distinctiveness, so its position in the distribution is directly comparable to the per-reporter bars.

In [ ]:
all_combined_mean = all_combined.mean()

fig, ax = plt.subplots(figsize=(5, 5))
counts, bin_edges, _ = ax.hist(stats["mean"], bins=30, color="grey", edgecolor="black", label="reporters")
bin_width = bin_edges[1] - bin_edges[0]

ax.bar(
    all_combined_mean,
    1,
    width=bin_width,
    color="black",
    edgecolor="black",
    align="center",
    label=f"all reporters combined = {all_combined_mean:.3f}",
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlabel("mean distinctiveness")
ax.set_ylabel("Number of Reporters")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "reporter_distinctiveness_histogram.svg", bbox_inches="tight")
plt.show()

## Per-gene distinctiveness heatmaps

Three heatmaps sharing a single clustered gene-KO order on the y axis:

1. all genes × all fluorescent markers (Phase excluded),
2. a single Phase column,
3. a single all-fluorescence-combined column (the `all_combined` reference).

The gene (row) order is derived once by hierarchically clustering the
all-markers-except-Phase matrix (Phase is **not** used for clustering), then
reused for every plot so the rows line up across all three.

In [ ]:
# Split Phase off the marker matrix; `all_combined` was popped at load time.
phase_col = df["Phase"]
markers_df = df.drop(columns=["Phase"])
print(f"{markers_df.shape[1]} fluorescent markers (Phase excluded), {markers_df.shape[0]} genes")

# Cluster the gene rows on the markers-only matrix (Phase NOT used for clustering);
# cluster the marker columns independently. This row order is shared by all plots.
markers_filled = markers_df.fillna(np.nanmean(markers_df.to_numpy()))
gene_row_order = leaves_list(
    linkage(pdist(markers_filled.to_numpy(), metric="correlation"), method="average")
)
marker_col_order = leaves_list(
    linkage(pdist(markers_filled.to_numpy().T, metric="correlation"), method="average")
)
gene_index = markers_df.index[gene_row_order]

markers_ord = markers_df.iloc[gene_row_order, :].iloc[:, marker_col_order]
phase_ord = phase_col.loc[gene_index]
combined_ord = all_combined.loc[gene_index]

In [ ]:
# Plot 1: all genes x all fluorescent markers (Phase excluded), rows clustered WITHOUT Phase.
n_rep, n_gene = markers_ord.shape[1], markers_ord.shape[0]
fig, ax = plt.subplots(figsize=(max(12, 0.32 * n_rep), max(12, 0.025 * n_gene)))
im = ax.imshow(markers_ord.to_numpy(), aspect="auto", cmap="Reds", vmin=0.0, vmax=1.0)

ax.set_xticks(range(n_rep))
ax.set_xticklabels(markers_ord.columns, rotation=90, fontsize=8)
ax.set_yticks(range(n_gene))
ax.set_yticklabels(markers_ord.index, fontsize=1.5)
ax.set_xlabel("reporter")
ax.set_ylabel(f"gene (n={n_gene}, rows clustered WITHOUT Phase)")

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.04)
cbar.set_label("distinctiveness")

ax.set_title(
    f"Per-gene distinctiveness (fluorescent markers, Phase excluded)  "
    f"[{n_rep} reporters x {n_gene} genes]"
)
fig.savefig(FIGURES_DIR / "distinctiveness_markers_no_phase.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Plot 2: single Phase column (same gene order as Plot 1).
n_gene_p = phase_ord.shape[0]
fig, ax = plt.subplots(figsize=(3, max(12, 0.025 * n_gene_p)))
im = ax.imshow(phase_ord.to_numpy()[:, None], aspect="auto", cmap="Reds", vmin=0.0, vmax=1.0)

ax.set_xticks([0])
ax.set_xticklabels(["Phase"], rotation=90, fontsize=8)
ax.set_yticks(range(n_gene_p))
ax.set_yticklabels(phase_ord.index, fontsize=1.5)
ax.set_xlabel("reporter")
ax.set_ylabel(f"gene (n={n_gene_p}, rows clustered WITHOUT Phase)")

cbar = fig.colorbar(im, ax=ax, fraction=0.08, pad=0.1)
cbar.set_label("distinctiveness")

ax.set_title(f"Per-gene distinctiveness (Phase)  [n={n_gene_p} genes]")
fig.savefig(FIGURES_DIR / "distinctiveness_phase.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Plot 3: single all-fluorescence-combined column (same gene order as Plot 1).
n_gene_c = combined_ord.shape[0]
fig, ax = plt.subplots(figsize=(3, max(12, 0.025 * n_gene_c)))
im = ax.imshow(combined_ord.to_numpy()[:, None], aspect="auto", cmap="Reds", vmin=0.0, vmax=1.0)

ax.set_xticks([0])
ax.set_xticklabels(["all fluorescence combined"], rotation=90, fontsize=8)
ax.set_yticks(range(n_gene_c))
ax.set_yticklabels(combined_ord.index, fontsize=1.5)
ax.set_xlabel("reporter")
ax.set_ylabel(f"gene (n={n_gene_c}, rows clustered WITHOUT Phase)")

cbar = fig.colorbar(im, ax=ax, fraction=0.08, pad=0.1)
cbar.set_label("distinctiveness")

ax.set_title(f"All-fluorescence-combined distinctiveness  [n={n_gene_c} genes]")
fig.savefig(FIGURES_DIR / "distinctiveness_all_fluorescence_combined.svg", bbox_inches="tight")
plt.show()